# Etude de l'impact du choix du LLM sur les persformances
Présentation de ce que l'on veut faire ici.

In [1]:
from google import genai
from google.oauth2 import service_account
from io import StringIO
import numpy as np
import re
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
import hashlib
import requests
from pathlib import Path
import tomllib
import pandas as pd
from typing import Dict, Any
from tqdm import tqdm
import plotly.express as px
from difflib import SequenceMatcher
import plotly.graph_objects as go
from plotly.subplots import make_subplots


## 1. Initialisation
Nous commençons tout d'abord par importer les bibliothèques nécessaires et initialiser le client permettant d'accéder à un LLM via VertexAI.

In [2]:

# 1. Path jusqu'au fichier de clé JSON
KEY_PATH = "projet-tutore-uga-98c5d56c8a8e.json"

# 2. Credentials object
SCOPES = ["https://www.googleapis.com/auth/cloud-platform"]
CREDENTIALS = service_account.Credentials.from_service_account_file(
    KEY_PATH,
    scopes=SCOPES
    )

# 3.Paramètres du projet
PROJECT_ID = "projet-tutore-uga"
REGION = "global"

# 4. Initialisation du client GenAI
client = genai.Client(
    vertexai=True, project=PROJECT_ID, location=REGION, credentials=CREDENTIALS
)

# 5. Modèle
MODEL_ID = "gemini-3-pro-preview"

## 2. Jeu de données
Récupération des données

In [3]:
# Define the URL and expected SHA-1 checksum
url = "https://www.data.gouv.fr/fr/datasets/r/bc085888-e6bd-445d-b3f4-632190c29e3f"
expected_sha1 = "90540350af64eb61f8a9823c83468934b19634c1"

# Define the target directory and file path
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)
file_path = data_dir / "01_251021_AvisFiscalite.csv"

# Check if the file already exists
if file_path.exists():
    print(f"File already exists: {file_path}")
else:
    # Download the file if it doesn't exist
    print(f"Downloading file: {file_path}")
    response = requests.get(url)
    response.raise_for_status()  # Raise an error for bad HTTP responses
    file_path.write_bytes(response.content)

    # Verify the SHA-1 checksum
    sha1 = hashlib.sha1()
    with file_path.open("rb") as f:
        while chunk := f.read(8192):
            sha1.update(chunk)
    calculated_sha1 = sha1.hexdigest()
    if calculated_sha1 == expected_sha1:
        print("SHA-1 checksum verified successfully.")
    else:
        print(f"SHA-1 checksum mismatch! Expected: {expected_sha1}, Got: {calculated_sha1}")

File already exists: ..\data\01_251021_AvisFiscalite.csv


In [4]:
df = pd.read_csv(file_path, sep=",")
col_name = "QUXVlc3Rpb246MTYz - Que faudrait-il faire pour rendre la fiscalité plus juste et plus efficace ?"
df_contrib = df[['authorId', col_name]].rename(
    {col_name:'contribution'},
    axis=1)
df_contrib = df_contrib.dropna(subset=['contribution'])

C:\Users\garan\AppData\Local\Temp\ipykernel_21896\299903361.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, sep=",")


In [5]:
df_contrib.head()

,authorId,contribution
3,VXNlcjpjNDY0ZjllMy0xZDk4LTExZTktOTRkMi1mYTE2M2...,Repartir les richesses. suppression de la tax...
4,VXNlcjo3MDdkM2IzOC0xZDYxLTExZTktOTRkMi1mYTE2M2...,"Les droits soient automatiques, comme nos devo..."
7,VXNlcjoxZTNlOTExYi0xZTIwLTExZTktOTRkMi1mYTE2M2...,réduire drastiquement la fraude fiscale. Impos...
8,VXNlcjo1ODljMWRiMy0xZDVhLTExZTktOTRkMi1mYTE2M2...,diminuer le taux de prelevement pour les retra...
9,VXNlcjo0OTUzNmNmYy0xZTIwLTExZTktOTRkMi1mYTE2M2...,TOUT FRANÇAIS DEVRA PAYER L’IMPÔT QU’IL SOIT D...


## 3. Extraction avec le nouveau LLM

Fonctions utilitaires

In [6]:
class LLMBadCSV(Exception):
    pass

# Fonction pour nettoyer les balises de code et autres ajouts indésirables
def strip_code_fences(s: str) -> str:
    # Supprime les balises de code Markdown
    s = s.strip()
    s = re.sub(r"^```[a-zA-Z]*\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    # Supprime tout préfixe avant "CSV:" si ça arrive
    idx = s.find("CSV:")
    if idx != -1:
        s = s[idx:]
    return s

# Fonction pour extraire le bloc CSV du texte retourné par le LLM
def extract_csv_block(s: str) -> str:
    s = strip_code_fences(s)
    # On vérifie si la sortie commence par "CSV:" comme on s'y attendd'après le prompt
    if s.startswith("CSV:"):
        return s[len("CSV:"):] # On retourne tout ce qui suit "CSV:"
    # Sinon, on tente de récupérer les lignes à partir de l'entête demandée dans le prompt
    m = re.search(r"(?mi)^description,type,syntax,semantic\s*$", s)
    if m:
        return s[m.start():] # On retourne tout à partir de l'entête
    return s  # Sinon on retourne tout le texte

# Fonction pour normaliser une ligne du CSV si les contraintes ne sont pas respectées
def normalize_row(row: Dict[str, Any]) -> Dict[str, Any]:
    # Normalise les valeurs attendues
    row["type"] = str(row.get("type","")).strip().lower()
    row["syntax"] = str(row.get("syntax","")).strip().lower()
    row["semantic"] = str(row.get("semantic","")).strip().lower()

    # Contraintes
    type_ok = {"statement", "proposition"}
    syntax_ok = {"positive", "negative"}
    semantic_ok = {"positive", "negative", "neutral"}

    # Normalisation si besoin
    if row["type"] not in type_ok:
        row["type"] = "statement" 
    if row["syntax"] not in syntax_ok:
        row["syntax"] = "positive"
    if row["semantic"] not in semantic_ok:
        row["semantic"] = "neutral"
    return row

# Fonction pour parser le CSV retourné par le LLM en DataFrame pandas
def parse_llm_csv(csv_text: str) -> pd.DataFrame:
    csv_text = csv_text.strip()
    if not csv_text.lower().startswith("description,type,syntax,semantic"):
        # Parfois le modèle met des espaces, on nettoie la première ligne
        lines = csv_text.splitlines()
        if lines:
            header = lines[0].replace(" ", "")
            if header.lower() == "description,type,syntax,semantic":
                lines[0] = "description,type,syntax,semantic"
                csv_text = "\n".join(lines)

    try:
        df = pd.read_csv(StringIO(csv_text), dtype=str, keep_default_na=False)
    except Exception as e:
        raise LLMBadCSV(f"CSV illisible: {e}")

    # Colonnes minimales
    expected_cols = ["description", "type", "syntax", "semantic"]
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise LLMBadCSV(f"Colonnes manquantes: {missing}")

    # Normalisation
    df = df[expected_cols].copy()
    df = df.apply(lambda r: pd.Series(normalize_row(r.to_dict())), axis=1)
    return df

def build_message(text: str) -> str:
    message = f"""
    But: extraire les idées principales DISTINCTES d'un texte pour analyse.

    Règles:
    1. N'utiliser QUE le contenu de la CONTRIBUTION.
    2. Extraire la liste des idées DISTINCTES et PRINCIPALES.
    - Chaque idée = une phrase claire, autonome, reformulée si nécessaire.
    3. Pour CHAQUE idée, annoter:
    - type: "statement" (constat) OU "proposition" (suggestion/recommandation/objectif).
    - syntax: "negative" si la phrase contient une négation explicite (ex.: "ne", "n'", "ne pas", "ne plus", "non"), sinon "positive".
    - semantic: "positive", "negative" ou "neutral" (valence sémantique).
    4. Sortie STRICTEMENT en CSV avec entête EXACTE:
    CSV:description,type,syntax,semantic
    - Délimiteur: virgule.
    - Chaque description entre guillemets doubles.
    - Échapper tout guillemet interne par duplication (ex.: ""chat"").
    - NE RIEN AJOUTER d'autre (pas de texte avant/après, pas de code fences).
    - Pas de lignes vides.

    Exemple: "Les chats retombent sur leurs pattes. Les chats n'ont pas neuf vies. Il faut mieux prendre soin des chats pour prolonger leur vie." 
    CSV:description,type,syntax,semantic
    "Les chats retombent sur leurs pattes",statement,positive,neutral
    "Les chats n'ont pas neuf vies",statement,negative,negative
    "Il faut mieux prendre soin des chats pour prolonger leur vie",proposition,positive,positive

    CONTRIBUTION:
    {text}
    """
    
    return message

# Fonction principale pour appeler le LLM et obtenir un DataFrame avec les idées extraites
def call_llm_return_df(text: str) -> pd.DataFrame:
    messages = build_message(text)
    response = client.models.generate_content(model=MODEL_ID, contents=messages)
    raw = response.text
    csv_block = extract_csv_block(raw)
    return parse_llm_csv(csv_block)

In [7]:
# Fonction pour itérer sur les contributions et extraire les idées en utilisant le LLM
def extract_ideas_from_df(df_contrib: pd.DataFrame,
                          text_col: str = "contribution",
                          id_col: str = "authorId") -> pd.DataFrame:
    rows = []

    for i, row in tqdm(df_contrib.iterrows(), total=len(df_contrib), desc="LLM extraction"):
        text = str(row[text_col]).strip()
        auth = row[id_col]
        if not text:
            continue
        try:
            ideas_df = call_llm_return_df(text)
        except Exception as e:
            # On enregistre une ligne "échec" minimale pour traçabilité
            ideas_df = pd.DataFrame([{
                "description": f"[PARSE_FAIL] {str(e)[:200]}",
                "type": "statement",
                "syntax": "positive",
                "semantic": "neutral"
            }])

        # Ajoute le contexte
        ideas_df = ideas_df.copy()
        ideas_df.insert(0, "authorId", auth)
        ideas_df.insert(1, "contrib_index", i)
        ideas_df.insert(2, "contribution", text)

        # Concatène les idées extraites
        concatenated_ideas = " || ".join(ideas_df["description"].tolist())
        rows.append({
            "authorId": auth,
            "contrib_index": i,
            "contribution": text,
            "ideas": concatenated_ideas,
            "type": "statement",  # Garder les valeurs par défaut pour type, syntax et semantic
            "syntax": "positive",
            "semantic": "neutral"
        })

    if not rows:
        return pd.DataFrame(columns=["authorId", "contrib_index", "contribution", "ideas", "type", "syntax", "semantic"])
    out = pd.DataFrame(rows)
    return out

In [8]:
out_path = data_dir / "extraction_idees_principales_Gemini.csv"
if out_path.exists():
    result = pd.read_csv(out_path)
    msg = f"L'extraction a déjà été réalisée, {out_path}"
    print(msg)
else:
    result = extract_ideas_from_df(df_contrib[0:200])
    out_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(out_path, index=False)
    print(f"Extraction saved to {out_path}")

L'extraction a déjà été réalisée, ..\data\extraction_idees_principales_Gemini.csv


In [9]:
result.head()

,authorId,contrib_index,contribution,ideas,type,syntax,semantic
0,VXNlcjpjNDY0ZjllMy0xZDk4LTExZTktOTRkMi1mYTE2M2...,3,Repartir les richesses. suppression de la tax...,Il faut répartir les richesses || Il faut supp...,statement,positive,neutral
1,VXNlcjo3MDdkM2IzOC0xZDYxLTExZTktOTRkMi1mYTE2M2...,4,"Les droits soient automatiques, comme nos devo...","Les droits doivent être automatiques, tout com...",statement,positive,neutral
2,VXNlcjoxZTNlOTExYi0xZTIwLTExZTktOTRkMi1mYTE2M2...,7,réduire drastiquement la fraude fiscale. Impos...,Il faut réduire drastiquement la fraude fiscal...,statement,positive,neutral
3,VXNlcjo1ODljMWRiMy0xZDVhLTExZTktOTRkMi1mYTE2M2...,8,diminuer le taux de prelevement pour les retra...,Diminuer le taux de prélèvement pour les retra...,statement,positive,neutral
4,VXNlcjo0OTUzNmNmYy0xZTIwLTExZTktOTRkMi1mYTE2M2...,9,TOUT FRANÇAIS DEVRA PAYER L’IMPÔT QU’IL SOIT D...,Tout Français devra payer l'impôt qu'il soit d...,statement,positive,neutral


## 4. Notation humaine, hallucinations et idées séparées
Score moyen, taux hallucination, taux idées séparées (selon longueur), boxplot du score.

In [10]:
df = pd.read_csv("../data/04_Extractions_idees_gemini_notes.csv")
df["score_humain"] = (
    df["score_humain"]
        .str.replace(",", ".", regex=False)
        .astype(float)
)
# Copie des colonnes d'hallucinations/d'idées invalides en version catégorielle
mapping = {0:"Non", 1:"Oui"}
df["pres_hallu"] = df["Hallucinations"].map(mapping)
df["pres_idees_inv"] = df["Idées_non_ind"].map(mapping)

# Nombre de tokens par contribution et par extractions (comptage simple par espaces)
df["nb_tokens_contrib"] = df["contribution"].apply(lambda x: len(str(x).split()))

# Autre options : catégories basées sur des seuils fixes
df["contrib_tokens_bins_fixe"] = pd.cut(df["nb_tokens_contrib"], bins=[0, 50, 100, 200, np.inf], labels=[
    "Très court (0-50)", "Court (51-100)", "Moyen (101-200)", "Long (+200)"
])

In [11]:
df_base = pd.read_csv("../data/01_251021_Notations200.csv")
# Calcul du score humain moyen
df_base["score_humain"] = df_base[["Garance", "Matthias", "Yannis"]].mean(axis=1)/10

# Copie des colonnes d'hallucinations/d'idées invalides en version catégorielle
mapping = {0:"Non", 1:"Oui"}
df_base["pres_hallu"] = df_base["Hallucinations"].map(mapping)
df_base["pres_idees_inv"] = df_base["Idées_non_ind"].map(mapping)

# Nombre de tokens par contribution et par extractions (comptage simple par espaces)
df_base["nb_tokens_contrib"] = df_base["contribution"].apply(lambda x: len(str(x).split()))
# Autre options : catégories basées sur des seuils fixes
df_base["contrib_tokens_bins_fixe"] = pd.cut(df_base["nb_tokens_contrib"], bins=[0, 50, 100, 200, np.inf], labels=[
    "Très court (0-50)", "Court (51-100)", "Moyen (101-200)", "Long (+200)"
])

In [34]:
# Ajouter une colonne modèle
df_base["Modèle"] = "Llama 8B"
df["Modèle"] = "Gemini 3 Pro"

# Concaténer
df_compare = pd.concat([df_base, df])
df_compare = df_compare.reset_index(drop=True)

In [35]:
fig = px.box(
    df_compare,
    x="Modèle",
    y="score_humain",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    }
)
fig.update_layout(
    title="<b>Répartition des scores humains par modèle</b>",
    xaxis_title="Modèles",
    yaxis_title="Score humain",
    width=750,
    height=450
)
fig.show()

In [63]:
df["score_humain"].mean()


np.float64(0.8115000000000001)

In [64]:
df_base["score_humain"].mean()

np.float64(0.5381666666666667)

In [36]:
# Graphique combiné
fig = px.histogram(
    df_compare,
    x="contrib_tokens_bins_fixe",
    y="score_humain",
    histfunc="avg",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    },
    barmode="group",   # barres côte à côte
    category_orders={
        "contrib_tokens_bins_fixe": [
            "Très court (0-50)",
            "Court (51-100)",
            "Moyen (101-200)",
            "Long (+200)"
        ]
    }
)

fig.update_layout(
    title="<b>Comparaison des scores humains par catégorie de longueur</b>",
    xaxis_title="Catégories de longueur (en nombre de tokens)",
    yaxis_title="Score humain moyen",
    width=750,
    height=450
)

fig.update_traces(
    hovertemplate="Score humain = %{y:.2f}<extra></extra>"
)

fig.show()

In [37]:
(df["pres_hallu"] == "Oui").value_counts()

pres_hallu
False    199
True       1
Name: count, dtype: int64

In [38]:
(df_base["pres_hallu"] == "Oui").value_counts()

pres_hallu
False    134
True      66
Name: count, dtype: int64

In [39]:
(df["pres_idees_inv"] == "Oui").value_counts()

pres_idees_inv
False    176
True      24
Name: count, dtype: int64

In [40]:
(df_base["pres_idees_inv"] == "Oui").value_counts()

pres_idees_inv
False    134
True      66
Name: count, dtype: int64

## 5. Evaluation sur la pipeline

QualIT

In [41]:
embed_model = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(embed_model)
emb_contrib = model.encode(
    df_compare["contribution"].tolist(),
    device="cpu",
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)
emb_ideas = model.encode(
    df_compare["ideas_text"].fillna("").tolist(),
    device="cpu",
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)
qualit_scores = np.sum(emb_contrib * emb_ideas, axis=1)
df_compare["C"] = qualit_scores

In [42]:
fig = px.box(
    df_compare,
    x="Modèle",
    y="C",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    }
)
fig.update_layout(
    title="<b>Répartition des scores QualIT par modèle</b>",
    xaxis_title="Modèles",
    yaxis_title="Score QualIT",
    width=750,
    height=450
)
fig.show()

ROUGE

In [43]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
def calc_rouge(row):
    try:
        scores = scorer.score(str(row["ideas_text"]), str(row["contribution"]))
        return pd.Series({
            "rouge_score_1gram": scores["rouge1"].fmeasure,
            "rouge_score_L": scores["rougeL"].fmeasure
        })
    except Exception:
        return pd.Series({"rouge_score_1gram": 0.0, "rouge_score_L": 0.0})
        print("a")
df_compare[["rouge_score_1gram", "rouge_score_L"]] = df_compare.apply(calc_rouge, axis=1)

In [68]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Création de la figure avec 2 sous-graphes
fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "<b>ROUGE 1-GRAMM</b>",
    "<b>ROUGE-L</b>",
    "<b>QualIT</b>"
))

# Ajout du premier boxplot (avec légende)
fig1 = px.box(
    df_compare,
    x="Modèle",
    y="rouge_score_1gram",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    }
)
for i, trace in enumerate(fig1.data):
    fig.add_trace(trace, row=1, col=1)
    if i == 0:  # On garde la légende seulement pour la première trace
        trace.showlegend = True
    else:
        trace.showlegend = False

# Ajout du deuxième boxplot (sans légende)
fig2 = px.box(
    df_compare,
    x="Modèle",
    y="rouge_score_L",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    }
)
for trace in fig2.data:
    trace.showlegend = False
    fig.add_trace(trace, row=1, col=2)

fig3 = px.box(
    df_compare,
    x="Modèle",
    y="C",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    }
)
for trace in fig3.data:
    trace.showlegend = False
    fig.add_trace(trace, row=1, col=3)

# Mise à jour de la légende commune et des axes
fig.update_layout(
    width=1000,
    height=450,
    legend_title_text='Modèle'
)
fig.update_xaxes(title_text="Modèles", row=1, col=1)
fig.update_xaxes(title_text="Modèles", row=1, col=2)
fig.update_xaxes(title_text="Modèles", row=1, col=3)
fig.update_yaxes(title_text="Score ROUGE 1-GRAMM", row=1, col=1)
fig.update_yaxes(title_text="Score ROUGE-L", row=1, col=2)
fig.update_yaxes(title_text="Score QualIT", row=1, col=3)

fig.show()


NLI base

In [45]:
# Jaccard et LCS ratio
_SENT_SPLIT = re.compile(r'(?<=[\.\?\!])\s+|\n+')
_WORD = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9']+")

NEG_MARKERS = {
    "ne", "n", "pas", "plus", "jamais", "aucun", "aucune", "sans", "ni", "rien", "personne"
}

STOPWORDS_FR_MINI = {
    # mini stoplist (évite de dépendre d'un package)
    "le","la","les","un","une","des","du","de","d","et","ou","à","a","au","aux",
    "en","dans","sur","pour","par","avec","sans","ce","cet","cette","ces",
    "que","qui","quoi","dont","où","est","sont","être","été","être","il","elle",
    "ils","elles","on","nous","vous","je","tu","se","sa","son","ses","leur","leurs",
    "mais","donc","car","si","comme","plus","moins","très"
}

def split_sentences(text: str):
    if not isinstance(text, str) or not text.strip():
        return []
    text = re.sub(r"\s+", " ", text.strip())
    return [s.strip() for s in _SENT_SPLIT.split(text) if s and s.strip()]

def tokenize(text: str):
    if not isinstance(text, str):
        return []
    toks = [t.lower() for t in _WORD.findall(text)]
    return toks

def content_tokens(text: str):
    toks = tokenize(text)
    return [t for t in toks if t not in STOPWORDS_FR_MINI and len(t) > 2]

def has_negation(text: str):
    toks = tokenize(text)
    return any(t in NEG_MARKERS for t in toks) or "n'" in text.lower()

def jaccard(a_tokens, b_tokens):
    A, B = set(a_tokens), set(b_tokens)
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)

def lcs_ratio(a: str, b: str):
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def parse_ideas_text(ideas_text: str):
    if not isinstance(ideas_text, str) or not ideas_text.strip():
        return []
    ideas = split_sentences(ideas_text)
    ideas = [x.strip() for x in ideas if len(x.strip()) > 3]
    return ideas

# Score NLI
def nli_lexical_scores(premise: str, ideas_text: str, alpha_contra: float = 0.8):
    premise_sents = split_sentences(premise)
    if not premise_sents:
        return np.nan, np.nan, np.nan

    ideas = parse_ideas_text(ideas_text)
    if not ideas:
        return np.nan, np.nan, np.nan

    contra_scores = []
    support_scores = []

    for h in ideas:
        h_tok = content_tokens(h)
        h_neg = has_negation(h)

        best_support = 0.0
        best_sent = None

        for ps in premise_sents:
            ps_tok = content_tokens(ps)
            s_j = jaccard(h_tok, ps_tok)
            s_l = lcs_ratio(h, ps)
            support = 0.6 * s_j + 0.4 * s_l
            if support > best_support:
                best_support = support
                best_sent = ps

        support_scores.append(best_support)

        if best_sent is None:
            contra_scores.append(0.0)
        else:
            ps_neg = has_negation(best_sent)
            mismatch = 1.0 if (h_neg != ps_neg) else 0.0
            contra = mismatch * (1.0 - best_support) * alpha_contra
            contra_scores.append(contra)

    support_mean = float(np.mean(support_scores)) if support_scores else np.nan
    contra_mean = float(np.mean(contra_scores)) if contra_scores else np.nan
    final = float(np.clip(support_mean - contra_mean, 0.0, 1.0)) if np.isfinite(support_mean) and np.isfinite(contra_mean) else np.nan

    return support_mean, contra_mean, final

In [46]:
# Application à nos données
df_compare["NLI_support"] = np.nan
df_compare["NLI_contra"] = np.nan
df_compare["NLI_final"] = np.nan

for i in range(len(df_compare)):
    premise = str(df_compare.loc[i, "contribution"]) if "contribution" in df_compare.columns else ""
    ideas_text = str(df_compare.loc[i, "ideas_text"]) if "ideas_text" in df_compare.columns else ""
    s, c, f = nli_lexical_scores(premise, ideas_text, alpha_contra=0.8)
    df_compare.loc[i, "NLI_support"] = s
    df_compare.loc[i, "NLI_contra"] = c
    df_compare.loc[i, "NLI_final"] = f

df_compare[["NLI_support", "NLI_contra", "NLI_final"]].describe()

,NLI_support,NLI_contra,NLI_final
count,399.000000,399.000000,399.000000
mean,0.461120,0.140428,0.400992
std,0.260666,0.251866,0.306104
min,0.000000,0.000000,0.000000
25%,0.256204,0.000000,0.090524
50%,0.450000,0.000000,0.398041
75%,0.631736,0.180870,0.619738
max,1.000000,0.773922,1.000000


In [49]:
fig = px.box(
    df_compare,
    x="Modèle",
    y="NLI_final",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    }
)
fig.update_layout(
    title="<b>Répartition des scores NLI_final par modèle</b>",
    xaxis_title="Modèles",
    yaxis_title="Score NLI_final",
    width=750,
    height=450
)
fig.show()

NLI Roberta

In [51]:
df_compare.to_csv("../data/04_Comparaison_LLM.csv", index=False)

In [52]:
df_compare = pd.read_csv("../data/04_Comparaison_LLM_with_NLI_Roberta.csv")

In [53]:
#score nli entailment/contradiction/entailment-contradiction
df_compare["nli_entailment_minus_contradiction"] = (
    df_compare["nli_entailment"] - df_compare["nli_contradiction"]
)

correlation_entailment = df_compare["score_humain"].corr(df_compare["nli_entailment"])
print(f"Corrélation entre le score humain moyen et le score NLI entailment : {correlation_entailment:.2f}")

correlation_contradiction = df_compare["score_humain"].corr(df_compare["nli_contradiction"])
print(f"Corrélation entre le score humain moyen et le score NLI contradiction : {correlation_contradiction:.2f}")

correlation_ent_minus_contra = df_compare["score_humain"].corr(
    df_compare["nli_entailment_minus_contradiction"]
)
print(f"Corrélation entre le score humain moyen et le score NLI difference : {correlation_ent_minus_contra:.2f}")

Corrélation entre le score humain moyen et le score NLI entailment : 0.54
Corrélation entre le score humain moyen et le score NLI contradiction : -0.15
Corrélation entre le score humain moyen et le score NLI difference : 0.51


In [72]:
# Création de la figure avec 3 sous-graphes
fig = make_subplots(rows=1, cols=4, subplot_titles=(
    "<b>NLI_final</b>",
    "<b>NLI entailment</b>",
    "<b>NLI contradiction</b>",
    "<b>NLI difference</b>"
))

fig1 = px.box(
    df_compare,
    x="Modèle",
    y="NLI_final",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    }
)
for i, trace in enumerate(fig1.data):
    fig.add_trace(trace, row=1, col=1)
    if i == 0:  # Gestion de la légende, on garde que la première
        trace.showlegend = True
    else:
        trace.showlegend = False
    
fig2 = px.box(
    df_compare,
    x="Modèle",
    y="nli_entailment",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    }
)
for trace in fig2.data:
    trace.showlegend = False
    fig.add_trace(trace, row=1, col=2)

fig3 = px.box(
    df_compare,
    x="Modèle",
    y="nli_contradiction",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    }
)
for trace in fig3.data:
    trace.showlegend = False
    fig.add_trace(trace, row=1, col=3)

# Ajout du troisième boxplot (sans légende)
fig4 = px.box(
    df_compare,
    x="Modèle",
    y="nli_entailment_minus_contradiction",
    color="Modèle",
    color_discrete_map={
        "Llama 8B": "#466784",
        "Gemini 3 Pro": "#e1552a"
    }
)
for trace in fig4.data:
    trace.showlegend = False
    fig.add_trace(trace, row=1, col=4)
    

# Mise à jour de la légende commune et des axes
fig.update_layout(
    width=1500,
    height=450,
    legend_title_text='Modèle'
)
fig.update_xaxes(title_text="Modèles", row=1, col=1)
fig.update_xaxes(title_text="Modèles", row=1, col=2)
fig.update_xaxes(title_text="Modèles", row=1, col=3)
fig.update_xaxes(title_text="Modèles", row=1, col=4)
fig.update_yaxes(title_text="Score NLI_final", row=1, col=1)
fig.update_yaxes(title_text="Score NLI entailment", row=1, col=2)
fig.update_yaxes(title_text="Score NLI contradiction", row=1, col=3)
fig.update_yaxes(title_text="Score NLI difference", row=1, col=4)

fig.show()



## 6. Global